# Gene essentiality

In [ ]:
# Setup. On Google Colab this downloads the course files and installs the dependencies (~2 min).
# Locally (conda env from environment.yml) it only moves to the course folder so `data/` is found.
import os, sys

if "google.colab" in sys.modules:
    if not os.path.isdir("/content/cell-factory-design-course"):
        !git clone -q --depth 1 https://github.com/lab-biotek-bio-ugm/cell-factory-design-course /content/cell-factory-design-course
        %pip install -q --progress-bar off -r /content/cell-factory-design-course/requirements.txt
    from google.colab import output
    output.enable_custom_widget_manager()  # needed for Escher maps

for folder in (".", "..", "/content/cell-factory-design-course"):
    if os.path.isdir(os.path.join(folder, "data")):
        os.chdir(folder)
        break

## Overview

**Learning objectives.** After this notebook you can:

1. Simulate single gene knockouts and explain how gene-protein-reaction (GPR) rules translate a gene deletion into blocked reactions.
2. Classify genes as essential, growth-reducing or neutral from simulated growth rates.
3. Compare essentiality across growth conditions (carbon sources).
4. Determine which biomass precursors depend on which genes.

**Estimated time:** 45 min (20 min guided, 25 min exercises).

**Prerequisites:** notebooks 02 (GPR rules) and 05 (`with model:` contexts, media).

### Background

A gene is **essential** if the organism cannot grow without it under a given condition. Models predict essentiality through their **GPR rules**: Boolean expressions that link genes to reactions. `and` joins subunits of an enzyme complex (all are needed), `or` joins isozymes (any one is enough). Knocking out a gene sets it to `False`; every reaction whose rule then evaluates to `False` is blocked (bounds set to 0). For example, `b1723 or b3916` (the two PFK isozymes) survives the loss of either gene alone, while a single-gene rule like `b2926` (PGK) does not.

After the knockout we run FBA. Growth close to zero (in practice below a threshold, often 1-10 % of wild-type growth) means the gene is predicted essential.

**In silico versus in vivo.** Predictions can be compared with experimental knockout collections such as the *E. coli* Keio collection ([Baba et al., 2006](https://doi.org/10.1038/msb4100050)). Disagreements are informative: the medium in the model may differ from the experiment, the model lacks regulation and non-metabolic gene functions, GPR rules can be incomplete, and a model only covers the genes it contains (137 in the core model used here and 1366 in iJO1366 ([Orth et al., 2011](https://doi.org/10.1038/msb.2011.65)), while 4,288 genes were targeted when building the Keio collection).

**Biomass precursors.** The biomass reaction consumes a fixed recipe of precursors (amino acids or their precursors, nucleotides, cofactors, energy). Asking which knockouts prevent the production of each precursor separately explains *why* a gene is essential.

In [ ]:
import matplotlib.pyplot as plt
plt.style.use('ggplot')
import pandas as pd
from pandas import DataFrame
from cobra.io import read_sbml_model
%matplotlib inline
model = read_sbml_model('data/e_coli_core.xml.gz')

## Knocking out a single gene

In the model, exactly one gene is associated with the reaction [phosphoglycerate kinase](https://bigg.ucsd.edu/universal/reactions/PGK) (PGK), a glycolytic enzyme.

In [ ]:
model.reactions.PGK.genes

Checking the name of this gene in the model (the id is the Blattner number b2926) reveals that it is indeed [pgk](https://www.uniprot.org/uniprotkb/P0A799/entry).

In [ ]:
model.genes.b2926.name

Is [pgk](https://www.uniprot.org/uniprotkb/P0A799/entry) essential? Knock it out inside a `with` block (so the knockout is undone afterwards) and optimize:

In [ ]:
with model as pgk_mutant:
    pgk_mutant.genes.b2926.knock_out()
    print(pgk_mutant.slim_optimize())

Yes, it is: the growth rate is zero (or numerically indistinguishable from zero). PGK sits in lower glycolysis, which is needed both to break down glucose and, in reverse, for gluconeogenesis; the core model has no bypass.

## Perform all possible single gene deletions

A `for` loop iterates over all genes in the model and knocks out each one individually. The model is used as a context to undo every deletion before the next one. `slim_optimize(error_value=0.)` returns 0 instead of raising an error when a knockout makes the model infeasible. (cobrapy also offers `cobra.flux_analysis.single_gene_deletion`, which does the same in parallel.)

In [ ]:
essentiality = {}
for gene in model.genes:
    with model:
        gene.knock_out()
        essentiality[gene] = model.slim_optimize(error_value=0.)

In [ ]:
essentiality

In [ ]:
df = DataFrame.from_dict(essentiality, orient='index')
df = df.rename(columns={0: 'growth'})
df = df.sort_values('growth')

The histogram shows the distribution of growth rates: most knockouts have no or little effect, a group of genes is essential (growth 0), and some are in between.

In [ ]:
df.plot.hist()

In [ ]:
ax = df.plot.bar().axes
ax.figure.set_size_inches(16, 9)

The bar chart shows each gene, sorted by the growth rate of its knockout.

## Exercises

1. How many genes are in each category (essential and non-essential)? How many non-essential genes show a reduced growth rate?
2. Determine the essentiality of genes for all other available carbon sources (if this is too complicated, determine the essentiality profile for one other carbon source). The carbon source exchange reactions of the core model are listed in the cell below.

Hints:
* Some knockouts make the model infeasible. `slim_optimize(error_value=0.)` returns 0 in that case; with `model.optimize()` check `solution.status` instead.

In [ ]:
carbon_exchanges = ['EX_ac_e', 'EX_etoh_e', 'EX_for_e',
                    'EX_fru_e', 'EX_fum_e', 'EX_glc__D_e',
                    'EX_lac__D_e', 'EX_mal__L_e', 'EX_pyr_e', 'EX_succ_e']

In [ ]:
# Your code here


## Gene essentiality with respect to each biomass precursor

Instead of growth, we now ask for each biomass precursor separately whether the cell can still make it after a knockout. We add a demand reaction for the precursor, make that the objective, and knock out each gene.

In [ ]:
biomass_precursors = model.reactions.BIOMASS_Ecoli_core_w_GAM.reactants
len(biomass_precursors)

These are the metabolites consumed by the biomass reaction:

In [ ]:
biomass_precursors

In [ ]:
precursor_production = pd.DataFrame(columns=['knocked_gene', 'precursor', 'max_production'])
index = 0
for precursor in biomass_precursors:
    print('Determining essentiality profile for precursor: ' + precursor.id)
    with model:
        model.objective = model.add_boundary(precursor, type='demand')
        for gene in model.genes:
            with model:
                gene.knock_out()
                precursor_production.loc[index] = gene.name, precursor.id, model.slim_optimize(error_value=0.)
                index += 1

In [ ]:
precursor_production['can_produce'] = precursor_production['max_production'].apply(lambda x: 0 if x < 0.05 else 1)

Production below 0.05 mmol gDW⁻¹ h⁻¹ is counted as "cannot produce" (0), otherwise 1.

In [ ]:
precursor_production

In [ ]:
import plotly.graph_objs as go
from plotly.offline import init_notebook_mode, iplot
init_notebook_mode(connected=True)

In [ ]:
df = precursor_production.pivot(index='precursor', columns='knocked_gene', values='can_produce')
data = [go.Heatmap(z=df.values.tolist(), x=df.columns.tolist(), y=df.index.tolist())]
layout = go.Layout(xaxis=dict(title='knocked genes'), yaxis=dict(title='precursor'))
fig = go.Figure(data=data, layout=layout)
iplot(fig)

In the heatmap, each dark cell (0) is a precursor that can no longer be produced after knocking out that gene. Genes that block many precursors at once (dark columns) usually encode central enzymes; a knockout that blocks no single precursor but is still essential for growth points to a limitation in combining them, for example energy (ATP) supply.

## References

- Baba, Ara, Hasegawa et al. (2006). Construction of Escherichia coli K‐12 in‐frame, single‐gene knockout mutants: the Keio collection. *Molecular Systems Biology* 2:2006.0008. [doi.org/10.1038/msb4100050](https://doi.org/10.1038/msb4100050)
- Orth, Conrad, Na et al. (2011). A comprehensive genome‐scale reconstruction of Escherichia coli metabolism—2011. *Molecular Systems Biology* 7:535. [doi.org/10.1038/msb.2011.65](https://doi.org/10.1038/msb.2011.65)